<a href="https://colab.research.google.com/github/jumafernandez/modelado-gestion-datos-untref/blob/main/codigo/r_api_georef_graphql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Georef y GraphQL desde R

**Modelado y Gestión de Datos - UNTREF - 2026**

En este notebook usamos desde R dos APIs sin credenciales para ver cosas que no aparecen en los otros ejemplos. Con Georef, el servicio de normalización de datos geográficos de Argentina, recorremos una respuesta en páginas y normalizamos muchos nombres en un solo pedido POST. Con una API GraphQL de países, armamos una consulta que elige qué campos devolver.

Cada celda se ejecuta con **Shift + Enter** o con el botón ▶ de su izquierda.

## 1. Paquetes

httr2 arma y envía los pedidos; dplyr y tidyr ordenan los resultados. Colab ya los trae instalados.

In [ ]:
library(httr2)
library(dplyr)
library(tidyr)

## 2. Respuestas en páginas con Georef

Georef devuelve provincias, departamentos, municipios, localidades y calles con sus códigos oficiales. Cuando hay muchos resultados, los entrega en páginas: `max` indica cuántos por página e `inicio`, desde qué posición. Como la respuesta dice cuántos hay en total (`total`), se piden páginas hasta completarlos.

Traemos los municipios de la provincia de Buenos Aires, cuyo código es 06, de a 50 por página.

In [ ]:
GEOREF <- "https://apis.datos.gob.ar/georef/api"

municipios <- data.frame()
inicio <- 0
repeat {
  pagina <- request(GEOREF) |>
    req_url_path_append("municipios") |>
    req_url_query(provincia = "06", campos = "id,nombre", max = 50, inicio = inicio) |>
    req_perform() |>
    resp_body_json(simplifyVector = TRUE)

  municipios <- rbind(municipios, pagina$municipios)
  cat("desde la posición", inicio, "llegaron", nrow(pagina$municipios), "de", pagina$total, "\n")
  inicio <- inicio + 50
  if (inicio >= pagina$total) break
}

head(municipios)

Georef acepta hasta 5.000 resultados por página, así que acá alcanzaría con un solo pedido. Muchas APIs tienen límites más bajos, y la paginación es la forma de traer todo.

## 3. Normalizar nombres con un pedido POST a Georef

Un problema típico de la integración de datos es la misma provincia escrita de muchas formas. En esta encuesta inventada, la columna `provincia` la completó cada persona a mano.

In [ ]:
encuesta <- data.frame(
  provincia = c("Bs. As.", "CABA", "Cordoba", "Buenos Aires", "Capital Federal",
                "cba", "Mendoza", "Pcia. de Buenos Aires", "Ciudad de Buenos Aires", "Sgo. del Estero"),
  ingreso = c(850, 1200, 760, 910, 1350, 700, 640, 880, 1100, 520)
)

table(encuesta$provincia)

Con GET se consulta un nombre por vez, en la URL. Con POST, el pedido lleva un cuerpo en JSON, y Georef acepta en un solo pedido una lista de consultas. Armamos la lista con una consulta por fila de la encuesta; `req_body_json()` la convierte en el cuerpo JSON y cambia el método a POST.

In [ ]:
consultas <- list()
for (nombre in encuesta$provincia) {
  consultas[[length(consultas) + 1]] <- list(nombre = nombre, max = 1, campos = "id,nombre")
}

respuesta <- request(GEOREF) |>
  req_url_path_append("provincias") |>
  req_body_json(list(provincias = consultas)) |>
  req_perform()

respuesta

La respuesta trae un resultado por consulta, en el mismo orden. `unnest()` los abre en columnas; con `keep_empty = TRUE` conserva las consultas sin resultado, así las filas siguen alineadas con la encuesta.

In [ ]:
resultados <- resp_body_json(respuesta, simplifyVector = TRUE)$resultados
oficiales <- unnest(resultados["provincias"], provincias, keep_empty = TRUE)

encuesta$id_provincia <- oficiales$id
encuesta$provincia_oficial <- oficiales$nombre
encuesta

Ahora la encuesta tiene el código oficial de cada provincia, el mismo que usa el INDEC, y se puede cruzar con cualquier otra fuente que lo tenga. Las diez formas de escribir quedan en cinco provincias. Lo que la API no reconoce queda vacío (`NA`) y se corrige a mano: es parte del trabajo de limpieza.

In [ ]:
encuesta |>
  group_by(provincia_oficial) |>
  summarise(respuestas = n(), ingreso_medio = mean(ingreso))

## 4. Una consulta GraphQL que elige los campos

En GraphQL hay un único endpoint y las consultas viajan por POST, en un JSON con el campo `query`. La consulta dice exactamente qué campos devolver y, con alias como `pais: name`, con qué nombre. Pedimos los países de América del Sur (`SA`) con su capital, su moneda y sus idiomas a la [API de países](https://countries.trevorblades.com/) que aparece en las referencias de la presentación.

In [ ]:
consulta <- '
{
  continent(code: "SA") {
    countries {
      pais: name
      capital
      moneda: currency
      idiomas: languages {
        idioma: name
      }
    }
  }
}'

respuesta <- request("https://countries.trevorblades.com/") |>
  req_body_json(list(query = consulta)) |>
  req_perform()

paises <- resp_body_json(respuesta, simplifyVector = TRUE)$data$continent$countries
paises[, c("pais", "capital", "moneda")]

Los idiomas vienen anidados, una lista por país. Con `unnest()` quedan en filas, una por país e idioma, y se pueden contar. Con una API REST, armar esta tabla habría requerido un pedido por país o recibir muchos campos que no usamos.

In [ ]:
por_idioma <- unnest(paises, idiomas)
count(por_idioma, idioma, sort = TRUE)

Un detalle: en GraphQL, una consulta con errores suele responder con código 200. El problema se informa dentro del cuerpo, en el campo `errors`, así que hay que revisarlo.

In [ ]:
respuesta <- request("https://countries.trevorblades.com/") |>
  req_body_json(list(query = '{ continent(code: "SA") { poblacion } }')) |>
  req_perform()

resp_status(respuesta)
resp_body_json(respuesta)$errors[[1]]$message

---

## Desde tu computadora

El mismo código funciona en RStudio: los paquetes se instalan una sola vez con `install.packages(c("httr2", "dplyr", "tidyr"))`. La [documentación de Georef](https://datosgobar.github.io/georef-ar-api/) describe todos sus recursos, y en la [API de países](https://countries.trevorblades.com/) se pueden armar y probar consultas GraphQL antes de llevarlas a R.